# 4.6 Obstacle Investigation

Per-tile 2-D map of the labeled point cloud with all extracted clusters overlaid.  
**Click any cluster dot** on the map → top-view and side-view appear on the right.

Use the **Tile** dropdown to switch between tiles.

In [ ]:
import sys, io
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import numpy as np
import pandas as pd
import laspy
from PIL import Image as PILImage
import ipywidgets as widgets
from IPython.display import display

# Agg canvas used for the detail views without touching the widget backend
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

from config import CLUSTERS_DIR, LABELED_DIR

In [ ]:
inv = pd.read_csv(CLUSTERS_DIR / 'inventory.csv')
TILECODES = sorted(inv['tilecode'].unique().tolist())
print(f'Loaded {len(inv)} clusters across {len(TILECODES)} tile(s)')
print(inv.groupby('label')['cluster_idx'].count().rename('count').to_string())

In [ ]:
LABEL_NAMES = {
    0: 'Unknown', 1: 'Road', 9: 'Ground', 10: 'Building',
    30: 'Tree', 40: 'Car', 60: 'Street Light', 83: 'Large Container',
}
DOT_COLORS  = {0: '#aaaaaa', 30: '#44ee44', 40: '#ff8800', 60: '#44aaff', 83: '#dd44dd'}
DOT_DEFAULT = '#ffffff'

LABEL_PRIORITY = {10:8, 1:7, 30:6, 40:5, 60:4, 83:4, 79:3, 90:3, 9:2, 0:1}
BG_RGB = {
    -1:(0.07,0.07,0.07),  0:(0.22,0.22,0.22),  1:(0.75,0.20,0.20),
     9:(0.50,0.50,0.50), 10:(0.20,0.35,0.70), 30:(0.20,0.65,0.20),
    40:(1.00,0.50,0.10), 60:(1.00,0.95,0.20), 79:(0.80,0.40,0.00),
    83:(0.70,0.20,0.70), 90:(0.90,0.60,0.10),
}
GRID_RES = 0.25  # metres per pixel

In [ ]:
# ── tile loading + 2-D background grid ────────────────────────────────────────
_tile_cache = {}   # tilecode -> (xy, labels)
_bg_cache   = {}   # tilecode -> (rgb ndarray, extent)

def _load_tile_raw(tilecode):
    if tilecode in _tile_cache:
        return _tile_cache[tilecode]
    laz_path = LABELED_DIR / f'bgt_labeled_{tilecode}.laz'
    print(f'  Loading {laz_path.name}…', end=' ', flush=True)
    pc  = laspy.read(laz_path)
    xy  = np.column_stack([np.asarray(pc.x, np.float32),
                           np.asarray(pc.y, np.float32)])
    has = 'label' in pc.point_format.extra_dimension_names
    lbl = np.asarray(pc.label, np.int32) if has else np.zeros(len(xy), np.int32)
    print(f'{len(xy):,} pts')
    _tile_cache[tilecode] = (xy, lbl)
    return xy, lbl

def _make_bg(xy, labels):
    x, y  = xy[:,0], xy[:,1]
    x0, y0 = float(x.min()), float(y.min())
    xi = np.floor((x - x0) / GRID_RES).astype(np.int32)
    yi = np.floor((y - y0) / GRID_RES).astype(np.int32)
    nx, ny = int(xi.max())+1, int(yi.max())+1
    order = np.argsort(np.vectorize(lambda l: LABEL_PRIORITY.get(int(l), 0))(labels))
    grid  = np.full((ny, nx), -1, np.int32)
    grid[yi[order], xi[order]] = labels[order]
    rgb   = np.full((ny, nx, 3), BG_RGB[-1], np.float32)
    for lv, c in BG_RGB.items():
        m = grid == lv
        if m.any(): rgb[m] = c
    extent = [x0, x0 + nx*GRID_RES, y0, y0 + ny*GRID_RES]
    return rgb, extent

def _get_bg(tilecode):
    if tilecode not in _bg_cache:
        xy, lbl = _load_tile_raw(tilecode)
        _bg_cache[tilecode] = _make_bg(xy, lbl)
    return _bg_cache[tilecode]

In [ ]:
# ── detail renderer — uses FigureCanvasAgg directly, no backend switch needed ──
def _sa(ax):
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
    ax.tick_params(labelsize=6, colors='grey')

def render_detail(row):
    """Return PNG bytes for top-view (XY) + side-view (XZ) of a cluster."""
    try:
        npz = np.load(row['npz_path'])
    except Exception as e:
        fig = Figure(figsize=(9, 4), facecolor='#1a1a1a')
        FigureCanvasAgg(fig)
        ax = fig.add_subplot(111)
        _sa(ax)
        ax.text(0.5, 0.5, f'Cannot load NPZ:\n{e}', color='#cc4444',
                ha='center', va='center', transform=ax.transAxes)
        return _fig_bytes(fig)

    xyz  = npz['xyz_centered']
    hag  = npz.get('height_ag', None)
    if hag is None or np.all(np.isnan(hag)):
        hag = xyz[:,2] - xyz[:,2].min()
    hag  = np.nan_to_num(hag, nan=0.0)

    import matplotlib.cm as cm
    colors = cm.plasma(np.clip(hag, 0, 6) / 6.0)
    pt_sz  = max(1, min(10, 3000 // max(len(xyz), 1)))

    lbl    = int(row['label'])
    name   = LABEL_NAMES.get(lbl, f'Label {lbl}')
    title  = (f"#{int(row['cluster_idx'])}  {name}  ·  "
              f"{int(row['n_raw_pts']):,} pts  ·  "
              f"{float(row['area_m2']):.2f} m²  ·  "
              f"{row.get('label_source', '')}")

    fig = Figure(figsize=(10, 4.5), facecolor='#1a1a1a')
    FigureCanvasAgg(fig)
    fig.suptitle(title, color='white', fontsize=8)
    ax_t = fig.add_subplot(1, 2, 1)
    ax_s = fig.add_subplot(1, 2, 2)
    _sa(ax_t); _sa(ax_s)

    ax_t.scatter(xyz[:,0], xyz[:,1], c=colors, s=pt_sz, linewidths=0)
    ax_t.set_aspect('equal')
    ax_t.set_title('top view (XY)', color='#aaaaaa', fontsize=8)
    ax_t.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_t.set_ylabel('ΔY (m)', color='grey', fontsize=7)

    ax_s.scatter(xyz[:,0], xyz[:,2], c=colors, s=pt_sz, linewidths=0)
    ax_s.set_aspect('equal')
    ax_s.set_title('side view (XZ)', color='#aaaaaa', fontsize=8)
    ax_s.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_s.set_ylabel('height (m)', color='grey', fontsize=7)

    fig.tight_layout()
    return _fig_bytes(fig)

def _fig_bytes(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110,
                facecolor='#1a1a1a', bbox_inches='tight')
    buf.seek(0)
    return buf.read()

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── right-side output widgets ─────────────────────────────────────────────────
detail_img = widgets.Image(
    value=b'', format='png',
    layout=widgets.Layout(width='560px'),
)
info_html = widgets.HTML(
    value='<span style="color:#666;font-size:12px">click a cluster dot on the map</span>',
)

# ── interactive map figure ────────────────────────────────────────────────────
fig_map, ax_map = plt.subplots(figsize=(6, 6))
fig_map.patch.set_facecolor('#111111')
ax_map.set_facecolor('#111111')
for sp in ax_map.spines.values(): sp.set_edgecolor('#444')
ax_map.tick_params(colors='#777', labelsize=7)
fig_map.tight_layout(pad=0.5)

# scatter artists and their inventory rows, rebuilt per tile
_scatter_rows = []   # list of (scatter_artist, sub_dataframe)
_sel_marker,  = ax_map.plot([], [], 'w*', markersize=16, zorder=6,
                             markeredgecolor='#ff3333', markeredgewidth=1.2)


def _draw_tile(tilecode):
    """Redraw ax_map with background + cluster dots for the given tile."""
    _scatter_rows.clear()
    ax_map.cla()
    ax_map.set_facecolor('#111111')
    for sp in ax_map.spines.values(): sp.set_edgecolor('#444')
    ax_map.tick_params(colors='#777', labelsize=7)

    rgb, extent = _get_bg(tilecode)
    ax_map.imshow(rgb, origin='lower', extent=extent,
                  interpolation='nearest', aspect='equal')
    ax_map.set_xlim(extent[0], extent[1])
    ax_map.set_ylim(extent[2], extent[3])
    ax_map.set_title(tilecode, color='white', fontsize=9)
    ax_map.set_xlabel('X (m RD)', color='#777', fontsize=7)
    ax_map.set_ylabel('Y (m RD)', color='#777', fontsize=7)

    tile_inv = inv[inv['tilecode'] == tilecode]
    for lbl_code, grp in tile_inv.groupby('label'):
        color = DOT_COLORS.get(lbl_code, DOT_DEFAULT)
        name  = LABEL_NAMES.get(lbl_code, f'Label {lbl_code}')
        sc = ax_map.scatter(
            grp['centroid_x'].values, grp['centroid_y'].values,
            c=color, s=90, zorder=5, label=name,
            edgecolors='#111', linewidths=0.6,
            picker=True, pickradius=8,
        )
        _scatter_rows.append((sc, grp.reset_index(drop=True)))

    ax_map.legend(facecolor='#1e1e1e', labelcolor='white',
                  edgecolor='#444', fontsize=7, loc='upper right')
    fig_map.canvas.draw_idle()


# ── pick event ────────────────────────────────────────────────────────────────
def _on_pick(event):
    for sc, grp in _scatter_rows:
        if event.artist is sc:
            pt_idx = event.ind[0]
            row    = grp.iloc[pt_idx]
            # highlight
            _sel_marker.set_data([row['centroid_x']], [row['centroid_y']])
            fig_map.canvas.draw_idle()
            # update info + detail
            lbl  = int(row['label'])
            name = LABEL_NAMES.get(lbl, f'Label {lbl}')
            info_html.value = (
                f'<span style="color:#ccc;font-size:12px">'
                f'<b>#{int(row["cluster_idx"])}  {name}</b>  '
                f'{int(row["n_raw_pts"]):,} pts  ·  '
                f'{float(row["area_m2"]):.2f} m²</span>'
            )
            detail_img.value = render_detail(row)
            break

fig_map.canvas.mpl_connect('pick_event', _on_pick)


# ── tile dropdown ─────────────────────────────────────────────────────────────
tile_dd = widgets.Dropdown(
    options=TILECODES, value=TILECODES[0],
    description='Tile:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '40px'},
)

def _on_tile(change):
    if change['name'] == 'value':
        detail_img.value = b''
        info_html.value  = '<span style="color:#666;font-size:12px">loading…</span>'
        _draw_tile(change['new'])
        info_html.value  = '<span style="color:#666;font-size:12px">click a cluster dot on the map</span>'

tile_dd.observe(_on_tile)

# ── layout ────────────────────────────────────────────────────────────────────
right = widgets.VBox([info_html, detail_img])
display(widgets.VBox([
    tile_dd,
    widgets.HBox([fig_map.canvas, right],
                 layout=widgets.Layout(gap='20px', align_items='flex-start')),
]))

_draw_tile(TILECODES[0])

### Background colour legend
| Colour | Label |
|---|---|
| 🔴 Dark red | Road |
| 🔵 Blue | Building |
| ⚫ Mid-grey | Ground |
| 🟢 Green | Tree |
| 🟠 Orange | Car |
| 🟡 Yellow | Street light |

Cluster dots use the same colours. Click a dot to see its top-view and side-view on the right.